# Full-map experience collection with expanding square bands

Run in a Colab GPU runtime. Collect real observations from the frozen world-model planner and fit one new shared adapter.

- **Random starts in square strips.** Draw six seeded goals, one per vertical third in each room. Sample continuous starts from the legal portion of each newly added square strip. Half-widths are 6, 12, 18, 24, 30, 36, 42 and 48. The final width reaches the edge of the reset-safe domain (coordinates 8–56). Skip goal–band combinations with empty legal strips. The default schedule contains **270 rollout attempts**.
- **Collect unaided.** Raw latent MSE drives band 1. Each later band uses the previous checkpoint, frozen for that collection stage. Every rollout targets its final goal image.
- **Filter real steps.** Keep consecutive frames if the shortest legal route to the goal gets at least 0.5 units shorter. Keep up to eight qualifying transitions per rollout, spaced over the episode. Useful steps from failed rollouts can contribute. Report completed doorway crossings separately from goal success.
- **Fit cumulatively.** Start from identity. Keep earlier pairs eligible and fit 500 updates per nonempty band. Reuse the raw-cost normalization derived from band 1's replay. Reserved evaluation images are excluded from the fitting pairs.
- **Evaluate static geometry.** Compare raw and adapted MSE for eight withheld goals on a full-map grid. The rollout logs describe collection under the incoming checkpoint at each band.

Each finished episode and checkpoint is saved to Drive. Plan for several GB of space for captured observations and replay. To resume, set `RESUME_RUN_ID` to the printed run ID and keep the same numerical runtime.

In [ ]:
from google.colab import drive
from pathlib import Path
from datetime import datetime,timezone
import hashlib,json,shutil,signal,subprocess,sys,time
import torch

drive.mount('/content/drive')
assert torch.cuda.is_available(), 'Select a GPU runtime, then Run all.'
DRIVE=Path('/content/drive/MyDrive')
RESUME_RUN_ID=None  # Set to the printed ID to resume an interrupted run.
STARTS_PER_GOAL=6
RUN_ID=RESUME_RUN_ID or datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%S%fZ')
OUTPUT=DRIVE/'datasets/curriculum_cost_adapter/experience_square_v1'/RUN_ID
REPO=Path('/content/i-jepa-wm-experience-square')
REPO_URL='https://github.com/kmylonas/i-jepa-wm.git'
SOURCE_COMMIT='80c9cf0136cae13be48251c7bf0948d871cd4490'
DINO_ROOT=Path('/content/dino_wm')
DINO_COMMIT='0a9492fa12044b852ae9e001cc74604b79c8bb0c'
ENCODER_DRIVE=DRIVE/'checkpoints/IN1K-vit.h.14-300e.encoder-only.pth.tar'
WM_DRIVE=DRIVE/'datasets/wall_matched_fixed_multi_v3/fixed/world_model/best.pt'
ARTIFACTS=Path('/content/experience-square-artifacts')
ENCODER=ARTIFACTS/'encoder.pth.tar'
CHECKPOINT=ARTIFACTS/'world_model.pt'
ENCODER_SHA='3f37d8a7572e25c2dce9a3072c2fd8cd55ce32dc6edb92a8996b0e4cb140bf98'
WM_SHA='b0f3ccfefe0534d9f51af2d0767619913f48d0c323ebb323f0bcd4b800206c3f'

def sha256(path):
    h=hashlib.sha256()
    with Path(path).open('rb') as stream:
        for block in iter(lambda:stream.read(8*1024*1024),b''):h.update(block)
    return h.hexdigest()

def shell(command,cwd=None):
    command=list(map(str,command));print('$',' '.join(command),flush=True)
    with subprocess.Popen(command,cwd=cwd,stdout=subprocess.PIPE,stderr=subprocess.STDOUT,text=True,bufsize=1) as process:
        try:
            for line in process.stdout:print(line,end='',flush=True)
            code=process.wait()
        except BaseException:
            process.send_signal(signal.SIGINT)
            try:process.wait(timeout=10)
            except subprocess.TimeoutExpired:process.terminate()
            raise
    if code:raise subprocess.CalledProcessError(code,command)

print('GPU:',torch.cuda.get_device_name())
print('Run ID:',RUN_ID)
print('New experience curriculum results:',OUTPUT)

## Load the pinned models and renderer

Use the same frozen encoder, world model and CEM settings as the previous collection experiment.

In [ ]:
if not REPO.exists():shell(['git','clone','--branch','curriculum-cost-adapter','--single-branch',REPO_URL,REPO])
else:
    assert subprocess.check_output(['git','-C',str(REPO),'remote','get-url','origin'],text=True).strip()==REPO_URL
    assert not subprocess.check_output(['git','-C',str(REPO),'status','--porcelain'],text=True).strip(), 'Clone has local changes.'
    shell(['git','-C',REPO,'fetch','origin','curriculum-cost-adapter'])
shell(['git','-C',REPO,'checkout','--detach',SOURCE_COMMIT])
shell([sys.executable,'-m','pip','install','-q','gym==0.26.2','omegaconf','pyyaml','scipy','einops','matplotlib','pillow','imageio','imageio-ffmpeg'])
if not DINO_ROOT.exists():shell(['git','clone','https://github.com/gaoyuezhou/dino_wm.git',DINO_ROOT])
assert not subprocess.check_output(['git','-C',str(DINO_ROOT),'status','--porcelain','--untracked-files=no'],text=True).strip(), 'Renderer has tracked changes.'
shell(['git','-C',DINO_ROOT,'checkout',DINO_COMMIT])
ARTIFACTS.mkdir(parents=True,exist_ok=True)
for source,target,expected in ((ENCODER_DRIVE,ENCODER,ENCODER_SHA),(WM_DRIVE,CHECKPOINT,WM_SHA)):
    assert source.is_file(), f'Missing frozen artifact: {source}'
    if not target.exists() or sha256(target)!=expected:shutil.copy2(source,target)
    assert sha256(target)==expected, f'Unexpected checkpoint: {source}'

def command(operation, band=None):
    args=[sys.executable,'-m','src.mylonas_ijepa_wm.experience_square',operation,
          '--output-dir',OUTPUT,'--world-model-checkpoint',CHECKPOINT,
          '--encoder-checkpoint',ENCODER,'--dino-wm-root',DINO_ROOT,'--device','cuda',
          '--starts-per-goal',STARTS_PER_GOAL]
    if band is not None:
        args+=['--band',band]
    return args

def run_band(band):
    started=time.monotonic()
    shell(command('stage',band),cwd=REPO)
    print('Band minutes:',round((time.monotonic()-started)/60,2))

## Band 1 · square half-width 0–6

In [ ]:
run_band(1)

## Band 2 · square half-width 6–12

In [ ]:
run_band(2)

## Band 3 · square half-width 12–18

In [ ]:
run_band(3)

## Band 4 · square half-width 18–24

In [ ]:
run_band(4)

## Band 5 · square half-width 24–30

In [ ]:
run_band(5)

## Band 6 · square half-width 30–36

In [ ]:
run_band(6)

## Band 7 · square half-width 36–42

In [ ]:
run_band(7)

## Band 8 · square half-width 42–48

In [ ]:
run_band(8)

## Evaluate held-out goals

Each goal uses the same protected grid for every checkpoint and for raw MSE. Uphill edges are legal moves that shorten the route by at least 0.5 units while increasing latent cost.

In [ ]:
shell(command('compare'),cwd=REPO)
report=json.loads((OUTPUT/'comparison.json').read_text())
manifest=json.loads((OUTPUT/'experiment.json').read_text())['manifest']

In [ ]:
import pandas as pd

print('Collection goals:',[tuple(round(v,2) for v in g) for g in manifest['goals']])
print('Withheld goals:',report['goals'])
print('Held-out grid positions:',report['heldout_grid_points'])
rows=[]
for stage in report['stages']:
    for row in stage['collection']:
        route=row['case']['route']
        rows.append(dict(band=stage['band'],route=route,attempts=1,
                         successes=int(row['success']),steps=row['steps'],
                         opposite_room_reached=int(row['reached_other_room']),
                         doorway_passages=len(row['crossings']),eligible=row['eligible_pairs'],
                         retained=row['retained_pairs'],doorway_pairs=row['retained_doorway_pairs'],
                         retained_from_failures=row['retained_pairs'] if not row['success'] else 0))
collection=pd.DataFrame(rows)
print('By band')
display(collection.groupby('band').sum(numeric_only=True))
print('By band and route')
display(collection.groupby(['band','route']).sum(numeric_only=True))
collection.to_csv(OUTPUT/'collection_counts.csv',index=False)

rows=[]
for stage in report['stages']:
    for result in stage['by_goal']:
        for cohort, values in result['metrics'].items():
            for method, scores in values.items():
                rows.append(dict(band=stage['band'],goal=tuple(result['goal']),cohort=cohort,
                                 method=method,**scores))
table=pd.DataFrame(rows)
table[['correct','uphill']]*=100
table.to_csv(OUTPUT/'ranking_metrics.csv',index=False)
print('Mean rank accuracy across withheld goals (percent)')
scores=table[table.cohort.isin(['full_map','opposite_room','same_extent'])]
display(scores.pivot_table(index='band',columns=['cohort','method'],values='correct').round(2))
print('Mean wrong-direction local steps across withheld goals (percent)')
local=table[table.cohort.isin(['local','local_opposite_room'])]
display(local.pivot_table(index='band',columns=['cohort','method'],values='uphill').round(2))

## Inspect the full-map heatmaps

The star marks the withheld goal and gray marks the wall. Each panel uses its own unclipped scale. By default the notebook shows the checkpoint progression for one goal per room. Set `HEATMAP_GOALS` to inspect other goals.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.patches import Rectangle

HEATMAP_GOALS=[0,4]
axis=report['axis']
out=OUTPUT/'heatmaps'
out.mkdir(exist_ok=True)
for i in HEATMAP_GOALS:
    goal=report['goals'][i]
    panels=[('Legal path distance',report['truth_grid'][i]),('Raw latent MSE',report['raw_grid'][i])]
    panels.extend((f"Band {s['band']} · half-width {s['half_width']}",s['by_goal'][i]['grid_costs'])
                  for s in report['stages'])
    cols=3
    rows=(len(panels)+cols-1)//cols
    fig,axes=plt.subplots(rows,cols,figsize=(14,4.1*rows),constrained_layout=True)
    for ax in axes.flat[len(panels):]:
        ax.set_visible(False)
    for ax,(title,values) in zip(axes.flat,panels):
        field=np.asarray(values,dtype=float).reshape(len(axis),len(axis))
        image=ax.pcolormesh(axis,axis,field,cmap='viridis',shading='nearest',vmin=0)
        for top,bottom in [(axis[0],26),(34,axis[-1])]:
            ax.add_patch(Rectangle((29,top),6,bottom-top,facecolor='#424242'))
        ax.scatter(*goal,marker='*',c='black',edgecolor='white',s=110,zorder=4)
        ax.set(title=title,xlim=(axis[0],axis[-1]),ylim=(axis[-1],axis[0]),
               aspect='equal',xlabel='Ball x',ylabel='Ball y')
        fig.colorbar(image,ax=ax,shrink=.76)
    fig.suptitle(f'Withheld goal {tuple(goal)}')
    fig.savefig(out/f'goal_{i}.png',dpi=130)
    plt.show()
    plt.close(fig)